# Revision analyses

Reads `evaluation.json` files (run `python scripts/evaluate_results.py` first).

1. Main comparison per dataset (all methods, Whisper-tiny)
2. Threshold sensitivity
3. Success vs. query budget
4. SMACK: native targeted vs. untargeted adaptation (and targeted GATAS)
5. Generalization: GATAS and the waveform ablation across datasets and ASR models

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from results_loader import METHODS, DATASETS, ASR_MODELS, load_results, success_at_thresholds, success_at_budget

RESULTS = "../outputs/results"
MAIN_DATASETS = ["harvard", "librispeech"]
os.makedirs("_figures", exist_ok=True)

def summarize(records):
    rows = {}
    for m, recs in records.items():
        if not recs:
            continue
        rows[m] = {
            "n": len(recs),
            "success %": 100 * np.mean([r["success"] for r in recs]),
            "valid success %": 100 * np.mean([r["valid_success"] for r in recs]),
            "SET_OVERLAP": np.mean([r["set_overlap"] for r in recs]),
            "WER": np.mean([r["wer"] for r in recs]),
            "SBERT sim.": np.mean([r["sbert"] for r in recs]),
            "PESQ fitness": np.mean([r["pesq"] for r in recs]),
            "UTMOS": np.mean([r["utmos"] for r in recs]),
            "ref. ASR WER": np.mean([r["reference_wer"] for r in recs]),
            "runtime [s]": np.mean([r["elapsed_seconds"] for r in recs]),
            "queries": np.mean([r["queries"] for r in recs]),
        }
    return pd.DataFrame(rows).T

## 1. Main comparison

In [ ]:
main = {ds: load_results(RESULTS, dataset=ds, asr_model="whisper-tiny", methods=METHODS) for ds in MAIN_DATASETS}
for ds, records in main.items():
    print(f"=== {ds} ===")
    display(summarize(records).round(3))

## 2. Threshold sensitivity

GATAS / Waveform count as successful if any member of their final Pareto front meets the thresholds.
Note: with early stopping, GATAS stops once the default thresholds are met, so stricter thresholds underestimate it.

In [ ]:
so_grid = np.round(np.arange(0.0, 1.01, 0.1), 2)
pesq_grid = np.round(np.arange(0.05, 0.51, 0.05), 2)
wer_grid = np.round(np.arange(0.1, 1.01, 0.1), 2)

fig, axes = plt.subplots(len(MAIN_DATASETS), 3, figsize=(18, 5 * len(MAIN_DATASETS)), squeeze=False)
for row, (ds, records) in enumerate(main.items()):
    for m, recs in records.items():
        if not recs:
            continue
        so = [np.mean([success_at_thresholds(r, {"PESQ": 0.2, "SET_OVERLAP": t}) for r in recs]) for t in so_grid]
        pe = [np.mean([success_at_thresholds(r, {"PESQ": t, "SET_OVERLAP": 0.5}) for r in recs]) for t in pesq_grid]
        we = [np.mean([success_at_thresholds(r, {"PESQ": 0.2, "WER": ("min", t)}) for r in recs]) for t in wer_grid]
        axes[row, 0].plot(so_grid, so, marker="o", label=m)
        axes[row, 1].plot(pesq_grid, pe, marker="o", label=m)
        axes[row, 2].plot(wer_grid, we, marker="o", label=m)
    axes[row, 0].set_xlabel("SET_OVERLAP threshold (PESQ <= 0.2)")
    axes[row, 1].set_xlabel("PESQ fitness threshold (SET_OVERLAP <= 0.5)")
    axes[row, 2].set_xlabel("minimum WER (PESQ <= 0.2)")
    for ax in axes[row]:
        ax.set_ylabel(f"success rate ({ds})")
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)
    axes[row, 0].legend()
plt.tight_layout()
plt.savefig("_figures/threshold_sensitivity.pdf", bbox_inches="tight")

## 3. Success vs. query budget

In [ ]:
budgets = np.unique(np.logspace(1, 4, 30).astype(int))
fig, axes = plt.subplots(1, len(MAIN_DATASETS), figsize=(8 * len(MAIN_DATASETS), 5), squeeze=False)
for ax, (ds, records) in zip(axes[0], main.items()):
    for m, recs in records.items():
        if not recs:
            continue
        if recs[0].get("archive_history"):
            ax.plot(budgets, [np.mean([success_at_budget(r, b) for r in recs]) for b in budgets], label=m)
        else:
            ax.scatter([np.mean([r["queries"] for r in recs])], [np.mean([r["success"] for r in recs])], label=m, zorder=3)
    ax.set_xscale("log")
    ax.set_xlabel("attacked-ASR queries (PGD: gradient iterations)")
    ax.set_ylabel(f"success rate ({ds})")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend()
plt.savefig("_figures/success_vs_budget.pdf", bbox_inches="tight")

## 4. SMACK: native targeted vs. untargeted adaptation

In [ ]:
smack = load_results(RESULTS, dataset="harvard", asr_model="whisper-tiny",
                     methods=["SMACK_targeted", "GATAS_targeted", "SMACK", "GATAS"])
display(summarize(smack).round(3))
for m in ("SMACK_targeted", "GATAS_targeted"):
    recs = smack[m]
    if recs:
        print(f"{m}: target WER {np.mean([r['scores']['TARGET_WER'] for r in recs]):.3f}, "
              f"exact target match {100 * np.mean([r['scores']['TARGET_WER'] == 0 for r in recs]):.1f}%")

## 5. Generalization (GATAS and waveform ablation)

In [ ]:
rows = []
for asr in ASR_MODELS:
    for ds in DATASETS:
        records = load_results(RESULTS, dataset=ds, asr_model=asr, methods=["GATAS", "Waveform"])
        for m, recs in records.items():
            if recs:
                rows.append({"ASR": asr, "dataset": ds, "method": m,
                             "success %": 100 * np.mean([r["success"] for r in recs]),
                             "valid success %": 100 * np.mean([r["valid_success"] for r in recs]),
                             "WER": np.mean([r["wer"] for r in recs]),
                             "UTMOS": np.mean([r["utmos"] for r in recs]),
                             "queries": np.mean([r["queries"] for r in recs])})
generalization = pd.DataFrame(rows)
display(generalization.pivot_table(index=["ASR", "dataset"], columns="method",
                                   values=["success %", "valid success %", "UTMOS"]).round(2))